# Level 4: EXTRA HIGH  -  Proyek Tabular End-to-End yang Reproducible

**Estimasi waktu:** 4-5 jam

**Prasyarat:** Selesaikan `03_level_high.ipynb` terlebih dahulu.

## Tujuan Pembelajaran

Setelah menyelesaikan level ini, Anda akan mampu:

1. Membangun **pipeline end-to-end** untuk data tabular (numerik + kategorikal + missing values)
2. Menggunakan **ColumnTransformer** untuk preprocessing berbeda per jenis kolom
3. Menerapkan **template proyek** yang siap dipakai untuk CSV sendiri
4. Melakukan **clustering** (unsupervised learning) dengan K-Means
5. Memahami **reproducibility** dan pencatatan eksperimen
6. Memilih **threshold** berdasarkan kebutuhan bisnis
7. Membuat **laporan ringkasan proyek** data mining

---

## Teori: Dari Dataset Bersih ke Dunia Nyata

Di level sebelumnya, kita menggunakan dataset bawaan scikit-learn yang sudah bersih. Dunia nyata **sangat berbeda**:

| Aspek | Dataset Bawaan | Data Dunia Nyata |
|:------|:---------------|:-----------------|
| Missing values | Hampir tidak ada | 5-50% bisa missing |
| Tipe kolom | Semua numerik | Campuran numerik + kategorikal |
| Skala fitur | Relatif seragam | Sangat bervariasi |
| Kualitas | Sudah dibersihkan | Outlier, duplikat, inkonsisten |
| Feature engineering | Tidak perlu | Kunci sukses! |

### ColumnTransformer: Menangani Kolom Berbeda

```
DataFrame Asli:
  ┌──────────┬───────┬─────────┬────────┐
  │ skor_num │ harga │ kategori│ region │
  │  float   │ float │  str    │  str   │
  └──────────┴───────┴─────────┴────────┘
          │                    │
          v                    v
  ┌──────────────┐    ┌──────────────┐
  │ Numeric Pipe │    │ Categ Pipe   │
  │  Imputer     │    │  Imputer     │
  │  Scaler      │    │  OneHotEnc   │
  └──────────────┘    └──────────────┘
          │                    │
          └──────┬─────────────┘
                 v
         ColumnTransformer
         (menggabungkan keduanya)
                 │
                 v
              Model
```

### Reproducibility: Mengapa Penting?

Sebuah eksperimen data mining harus bisa **diulang** dengan hasil yang sama. Caranya:
1. Set `random_state` yang konsisten
2. Catat versi library yang digunakan
3. Simpan parameter model dan metrik
4. Dokumentasikan keputusan preprocessing

## 0. Persiapan

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from sklearn.datasets import make_classification, load_iris
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    ConfusionMatrixDisplay, average_precision_score,
    PrecisionRecallDisplay, f1_score, silhouette_score
)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 100

# Log versi library untuk reproducibility
import sklearn
print(f'numpy    : {np.__version__}')
print(f'pandas   : {pd.__version__}')
print(f'sklearn  : {sklearn.__version__}')
print(f'Random State: {RANDOM_STATE}')
print()
print(' Semua library berhasil dimuat!')

---

## BAGIAN A: Proyek Klasifikasi End-to-End (Supervised)

### 1. Simulasi Data Pelanggan (Churn Prediction)

Kita mensimulasikan skenario **prediksi churn pelanggan** dengan:
- Fitur **numerik**: skor engagement, pembayaran, dll.
- Fitur **kategorikal**: jenis plan, region
- **Missing values**: beberapa kolom memiliki data hilang
- **Kelas tidak seimbang**: ~65% stay vs ~35% churn

> ** Untuk proyek Anda sendiri:** Ganti bagian ini dengan `df = pd.read_csv('data_anda.csv')`

In [ ]:
# === Simulasi data pelanggan yang realistis ===
X_numeric, y_project = make_classification(
    n_samples=800,
    n_features=5,
    n_informative=3,
    n_redundant=1,
    weights=[0.65, 0.35],   # 65% stay, 35% churn
    flip_y=0.03,            # 3% noise
    random_state=RANDOM_STATE
)

rng = np.random.default_rng(RANDOM_STATE)

df = pd.DataFrame(X_numeric, columns=[
    'tenure_score',      # Lama berlangganan (terstandarisasi)
    'usage_score',       # Tingkat penggunaan produk
    'support_score',     # Frekuensi kontak support
    'payment_score',     # Skor pembayaran (keterlambatan, dll)
    'engagement_score'   # Skor engagement (login, aktivitas)
])

# Tambahkan fitur kategorikal
df['plan'] = rng.choice(['basic', 'pro', 'enterprise'], 
                         size=len(df), p=[0.50, 0.35, 0.15])
df['region'] = rng.choice(['barat', 'tengah', 'timur'], size=len(df))
df['channel'] = rng.choice(['online', 'offline', 'referral'], 
                            size=len(df), p=[0.60, 0.25, 0.15])

# Tambahkan missing values secara realistis
df.loc[rng.choice(df.index, size=35, replace=False), 'support_score'] = np.nan
df.loc[rng.choice(df.index, size=20, replace=False), 'engagement_score'] = np.nan
df.loc[rng.choice(df.index, size=10, replace=False), 'channel'] = np.nan

# Target
df['churn'] = y_project  # 0 = stay, 1 = churn

print('='*60)
print('DATASET: Prediksi Churn Pelanggan')
print('='*60)
print(f'Ukuran: {df.shape}')
print()
display(df.head(10))

### 2. EDA Cepat (Recap dari Level Light)

In [ ]:
# Kualitas data
print(' Kualitas Data:')
quality = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'non_null': df.count(),
    'missing': df.isna().sum(),
    'missing_pct': (df.isna().sum() / len(df) * 100).round(1),
    'unique': df.nunique(),
})
display(quality)

print(f'\n Duplikat: {df.duplicated().sum()}')

# Distribusi target
print(f'\n Distribusi Target (churn):')
churn_dist = df['churn'].value_counts()
print(f'  Stay  (0): {churn_dist[0]} ({churn_dist[0]/len(df)*100:.1f}%)')
print(f'  Churn (1): {churn_dist[1]} ({churn_dist[1]/len(df)*100:.1f}%)')

In [ ]:
# Visualisasi distribusi fitur numerik per kelas target
num_cols = df.select_dtypes(include='number').columns.drop('churn').tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flat

for i, col in enumerate(num_cols):
    for label, name, color in [(0, 'Stay', 'steelblue'), (1, 'Churn', 'tomato')]:
        subset = df[df['churn'] == label][col].dropna()
        axes[i].hist(subset, bins=25, alpha=0.6, label=name, color=color, edgecolor='white')
    axes[i].set_title(col, fontweight='bold')
    axes[i].legend(fontsize=8)

# Fitur kategorikal
cat_cols = df.select_dtypes(include='object').columns.tolist()
for i, col in enumerate(cat_cols, start=len(num_cols)):
    if i < len(axes.base.flat):
        ct = pd.crosstab(df[col], df['churn'], normalize='index')
        ct.plot(kind='bar', ax=axes[i], stacked=True, 
                color=['steelblue', 'tomato'], edgecolor='white')
        axes[i].set_title(col, fontweight='bold')
        axes[i].set_ylabel('Proporsi')
        axes[i].legend(['Stay', 'Churn'], fontsize=8)
        axes[i].tick_params(axis='x', rotation=0)

plt.suptitle('Distribusi Fitur per Kelas Target', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3. Membangun Pipeline End-to-End

### Teori: Anatomi Pipeline Lengkap

```
Full Pipeline:

  ┌─────────────────────────────────────────────┐
  │            ColumnTransformer                 │
  │                                              │
  │  Kolom Numerik         Kolom Kategorikal     │
  │  ┌────────────┐       ┌────────────────┐    │
  │  │ Imputer    │       │ Imputer        │    │
  │  │ (median)   │       │ (most_frequent)│    │
  │  ├────────────┤       ├────────────────┤    │
  │  │ Scaler     │       │ OneHotEncoder  │    │
  │  │ (standard) │       │                │    │
  │  └────────────┘       └────────────────┘    │
  └────────────────────┬────────────────────────┘
                       │
                       v
               ┌──────────────┐
               │    Model     │
               │ (classifier) │
               └──────────────┘
```

**Mengapa ini penting:**
1. **Imputer** mengisi missing values -> model tidak crash
2. **Scaler** untuk kolom numerik -> model berbasis jarak bekerja baik
3. **OneHotEncoder** untuk kolom kategorikal -> model bisa memproses teks
4. Semuanya di dalam **Pipeline** -> AMAN dari data leakage

In [ ]:
# === LANGKAH 1: Definisikan target dan fitur ===
target_col = 'churn'
X_project = df.drop(columns=target_col)
y_project = df[target_col]

# Identifikasi jenis kolom secara otomatis
numeric_features = X_project.select_dtypes(include='number').columns.tolist()
categorical_features = X_project.select_dtypes(include='object').columns.tolist()

print('Fitur Numerik    :', numeric_features)
print('Fitur Kategorikal:', categorical_features)
print()

# === LANGKAH 2: Buat sub-pipeline per jenis kolom ===
numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),   # Isi missing dengan median
    ('scaler', StandardScaler())                      # Standardisasi
])

categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),  # Isi missing dengan modus
    ('onehot', OneHotEncoder(handle_unknown='ignore',      # Handle kategori baru di test
                              sparse_output=False))        # Output array, bukan sparse
])

# === LANGKAH 3: Gabungkan dengan ColumnTransformer ===
preprocessor = ColumnTransformer([
    ('num', numeric_pipe, numeric_features),
    ('cat', categorical_pipe, categorical_features)
])

print(' Preprocessor siap!')
print(f'   Input: {len(numeric_features)} numerik + {len(categorical_features)} kategorikal')

In [ ]:
# === LANGKAH 4: Train/Test Split ===
X_train, X_test, y_train, y_test = train_test_split(
    X_project, y_project, 
    test_size=0.20, 
    stratify=y_project, 
    random_state=RANDOM_STATE
)

print(f'Training: {X_train.shape[0]} sampel')
print(f'Test    : {X_test.shape[0]} sampel')
print(f'Train churn rate: {y_train.mean():.1%}')
print(f'Test churn rate : {y_test.mean():.1%}')

In [ ]:
# === LANGKAH 5: Buat pipeline lengkap dengan beberapa model ===
pipelines = {
    'Baseline': Pipeline([
        ('preprocess', preprocessor),
        ('model', DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE))
    ]),
    'Logistic Regression': Pipeline([
        ('preprocess', preprocessor),
        ('model', LogisticRegression(
            max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE
        ))
    ]),
    'Random Forest': Pipeline([
        ('preprocess', preprocessor),
        ('model', RandomForestClassifier(
            n_estimators=100, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
        ))
    ]),
    'Gradient Boosting': Pipeline([
        ('preprocess', preprocessor),
        ('model', GradientBoostingClassifier(
            n_estimators=100, max_depth=3, random_state=RANDOM_STATE
        ))
    ]),
}

print(f' {len(pipelines)} pipeline siap untuk dibandingkan!')

In [ ]:
# === LANGKAH 6: Cross-validation untuk semua model ===
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print('='*65)
print('CROSS-VALIDATION RESULTS')
print('='*65)

cv_results = {}
for name, pipe in pipelines.items():
    # Validasi dengan beberapa metrik
    bal_acc_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='balanced_accuracy')
    ap_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='average_precision')
    
    cv_results[name] = {
        'bal_acc_mean': bal_acc_scores.mean(),
        'bal_acc_std': bal_acc_scores.std(),
        'ap_mean': ap_scores.mean(),
        'ap_std': ap_scores.std(),
        'bal_acc_scores': bal_acc_scores
    }
    
    print(f'\n{name}:')
    print(f'  Balanced Accuracy: {bal_acc_scores.mean():.3f} ± {bal_acc_scores.std():.3f}')
    print(f'  Average Precision: {ap_scores.mean():.3f} ± {ap_scores.std():.3f}')

# Tabel ringkasan
summary_df = pd.DataFrame({
    name: {'Bal Acc': f"{r['bal_acc_mean']:.3f} ± {r['bal_acc_std']:.3f}",
           'PR-AUC': f"{r['ap_mean']:.3f} ± {r['ap_std']:.3f}"}
    for name, r in cv_results.items()
}).T
print('\n Ringkasan:')
display(summary_df)

In [ ]:
# Visualisasi perbandingan
fig, ax = plt.subplots(figsize=(10, 5))

box_data = [cv_results[name]['bal_acc_scores'] for name in cv_results]
bp = ax.boxplot(box_data, labels=list(cv_results.keys()), patch_artist=True, vert=True)

colors = ['#d3d3d3', '#66b3ff', '#99ff99', '#ffcc99']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    
ax.set_ylabel('Balanced Accuracy')
ax.set_title('Perbandingan Model (CV Balanced Accuracy)', fontsize=13, fontweight='bold')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

### 4. Hyperparameter Tuning Model Terbaik

In [ ]:
# Tuning Random Forest (biasanya model terbaik untuk tabular data)
rf_tune_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestClassifier(
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
    ))
])

param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [5, 10, 15, None],
    'model__max_features': ['sqrt', 'log2'],
    'model__min_samples_leaf': [1, 3, 5],
}

total_combos = 2 * 4 * 2 * 3
print(f'Total kombinasi: {total_combos} × 5 fold = {total_combos * 5} fits')
print('Mohon tunggu...')

grid = GridSearchCV(
    estimator=rf_tune_pipeline,
    param_grid=param_grid,
    scoring='balanced_accuracy',
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)
grid.fit(X_train, y_train)

print(f'\n Parameter terbaik: {grid.best_params_}')
print(f'   CV Balanced Accuracy: {grid.best_score_:.3f}')

### 5. Evaluasi Akhir pada Data Test

In [ ]:
# Evaluasi final  -  HANYA SATU KALI pada data test
best_model = grid.best_estimator_

y_pred_final = best_model.predict(X_test)
y_proba_final = best_model.predict_proba(X_test)[:, 1]

print('='*60)
print('EVALUASI AKHIR  -  DATA TEST (HOLD-OUT)')
print('='*60)
print(f'Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred_final):.3f}')
print(f'Accuracy         : {accuracy_score(y_test, y_pred_final):.3f}')
print(f'PR-AUC           : {average_precision_score(y_test, y_proba_final):.3f}')
print(f'F1 (weighted)    : {f1_score(y_test, y_pred_final, average="weighted"):.3f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred_final, 
                            target_names=['Stay', 'Churn'], zero_division=0))

In [ ]:
# Confusion Matrix dan Precision-Recall Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_final,
    display_labels=['Stay', 'Churn'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('Confusion Matrix (Test)', fontsize=12, fontweight='bold')

PrecisionRecallDisplay.from_predictions(
    y_test, y_proba_final, ax=axes[1], name='Random Forest'
)
axes[1].set_title('Precision-Recall Curve (Test)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

### 6. Memilih Threshold Berdasarkan Kebutuhan Bisnis

### Teori: Threshold != 0.5 Selalu

Default threshold 0.5 jarang optimal. Pada churn prediction:
- **Mengirim promo retensi ke pelanggan yang tidak churn** = biaya kecil (FP)
- **Tidak mengirim promo ke pelanggan yang churn** = kehilangan pelanggan! (FN)

-> FN lebih mahal -> turunkan threshold -> tingkatkan recall

In [ ]:
# Analisis pengaruh threshold
from sklearn.metrics import precision_score, recall_score

thresholds = np.arange(0.1, 0.91, 0.05)
metrics_by_threshold = []

for t in thresholds:
    y_pred_t = (y_proba_final >= t).astype(int)
    metrics_by_threshold.append({
        'threshold': t,
        'precision': precision_score(y_test, y_pred_t, zero_division=0),
        'recall': recall_score(y_test, y_pred_t, zero_division=0),
        'f1': f1_score(y_test, y_pred_t, zero_division=0),
        'bal_acc': balanced_accuracy_score(y_test, y_pred_t),
    })

threshold_df = pd.DataFrame(metrics_by_threshold)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(threshold_df['threshold'], threshold_df['precision'], 'o-', label='Precision')
ax.plot(threshold_df['threshold'], threshold_df['recall'], 's-', label='Recall')
ax.plot(threshold_df['threshold'], threshold_df['f1'], '^-', label='F1-score')
ax.plot(threshold_df['threshold'], threshold_df['bal_acc'], 'D-', label='Balanced Acc')

ax.axvline(x=0.5, linestyle='--', color='gray', alpha=0.5, label='Default (0.5)')
best_f1_threshold = threshold_df.loc[threshold_df['f1'].idxmax(), 'threshold']
ax.axvline(x=best_f1_threshold, linestyle='--', color='red', alpha=0.5, 
           label=f'Best F1 ({best_f1_threshold:.2f})')

ax.set_xlabel('Threshold')
ax.set_ylabel('Skor')
ax.set_title('Pengaruh Threshold terhadap Metrik', fontsize=13, fontweight='bold')
ax.legend(loc='best')
ax.set_xlim(0.05, 0.95)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print(f' Threshold optimal untuk F1: {best_f1_threshold:.2f}')
print('   Jika ingin lebih banyak deteksi churn (recall tinggi), turunkan threshold.')
print('   Jika ingin lebih sedikit alarm palsu (precision tinggi), naikkan threshold.')

### 7. Feature Importance

In [ ]:
# Permutation importance
perm = permutation_importance(
    best_model, X_test, y_test,
    scoring='balanced_accuracy',
    n_repeats=15,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Dapatkan nama fitur yang tepat (termasuk one-hot encoded)
feature_names = numeric_features.copy()
# Ambil nama fitur dari OneHotEncoder
ohe = best_model.named_steps['preprocess'].named_transformers_['cat'].named_steps['onehot']
cat_feature_names = ohe.get_feature_names_out(categorical_features).tolist()
feature_names.extend(cat_feature_names)

imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance': perm.importances_mean,
    'std': perm.importances_std
}).sort_values('importance', ascending=False)

# Plot top features
top = imp_df.head(12).sort_values('importance')
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top['feature'], top['importance'], xerr=top['std'],
        color='steelblue', edgecolor='white', capsize=3)
ax.set_xlabel('Penurunan Balanced Accuracy')
ax.set_title('Feature Importance: Proyek Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(' Fitur dengan importance tertinggi paling memengaruhi prediksi model.')
print('   Gunakan ini untuk insight bisnis (bukan kausalitas!).')

---

## BAGIAN B: Clustering (Unsupervised Learning)

### Teori: Apa Itu Clustering?

Clustering mengelompokkan data tanpa label target. Berguna untuk:
- **Segmentasi pelanggan**: kelompokkan pelanggan berdasarkan perilaku
- **Deteksi anomali**: titik yang tidak masuk kelompok mana pun
- **Eksplorasi data**: menemukan struktur tersembunyi

### K-Means: Algoritma Clustering Paling Populer

```
Langkah K-Means:

1. Pilih K centroid acak         2. Assign setiap titik ke
                                    centroid terdekat
- *                       ooo     ...
     o o o  . .                     ooo     ...
     o o    . . .                   o       ...

3. Pindahkan centroid ke          4. Ulangi sampai konvergen
   rata-rata clusternya
     ooo    ...                     ooo     ...
     o*o    .*.                     o*o     .*.
     o      ...                     o       ...
```

### Silhouette Score: Seberapa Baik Cluster?

- Nilai **mendekati +1**: cluster terpisah baik
- Nilai **mendekati 0**: cluster tumpang tindih
- Nilai **negatif**: titik mungkin salah cluster

In [ ]:
# Clustering pada dataset Iris (tanpa label!)
iris = load_iris(as_frame=True)
X_cluster = iris.data.copy()

# Scaling diperlukan karena K-Means menggunakan jarak
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Coba berbagai jumlah cluster (Elbow Method + Silhouette)
k_range = range(2, 10)
inertias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow Method
axes[0].plot(k_range, inertias, 'o-', color='steelblue')
axes[0].set_xlabel('Jumlah Cluster (K)')
axes[0].set_ylabel('Inertia (within-cluster sum of squares)')
axes[0].set_title('Elbow Method', fontsize=12, fontweight='bold')

# Silhouette Score
axes[1].plot(k_range, silhouettes, 'o-', color='darkorange')
axes[1].set_xlabel('Jumlah Cluster (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis', fontsize=12, fontweight='bold')
best_k = list(k_range)[np.argmax(silhouettes)]
axes[1].axvline(x=best_k, linestyle='--', color='red', alpha=0.7, 
                label=f'Best K={best_k}')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f' Berdasarkan silhouette score, K={best_k} terlihat optimal.')
print('   Elbow method: cari "siku" dimana penurunan inertia mulai melambat.')

In [ ]:
# Visualisasi cluster optimal
km_final = KMeans(n_clusters=best_k, n_init=20, random_state=RANDOM_STATE)
cluster_labels = km_final.fit_predict(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Cluster yang ditemukan
scatter1 = axes[0].scatter(
    X_cluster['petal length (cm)'], X_cluster['petal width (cm)'],
    c=cluster_labels, cmap='viridis', alpha=0.7, edgecolors='w', s=60
)
axes[0].set_xlabel('Petal Length (cm)')
axes[0].set_ylabel('Petal Width (cm)')
axes[0].set_title(f'K-Means Clustering (K={best_k})', fontsize=12, fontweight='bold')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# Plot 2: Label sebenarnya (sebagai perbandingan)
scatter2 = axes[1].scatter(
    X_cluster['petal length (cm)'], X_cluster['petal width (cm)'],
    c=iris.target, cmap='viridis', alpha=0.7, edgecolors='w', s=60
)
axes[1].set_xlabel('Petal Length (cm)')
axes[1].set_ylabel('Petal Width (cm)')
axes[1].set_title('Label Sebenarnya (ground truth)', fontsize=12, fontweight='bold')
plt.colorbar(scatter2, ax=axes[1], label='Species')

plt.tight_layout()
plt.show()

print(' Perbandingan cluster K-Means vs label asli.')
print('   Perhatikan bahwa clustering TIDAK TAHU label yang benar,')
print('   tapi masih bisa menemukan struktur yang serupa!')

---

## BAGIAN C: Laporan Ringkasan Proyek

### Teori: Dokumentasi yang Baik

Setiap proyek data mining harus memiliki dokumentasi yang mencakup:

1. **Definisi masalah** dan metrik sukses
2. **Deskripsi data** dan keputusan preprocessing
3. **Hasil eksperimen** dan perbandingan model
4. **Limitasi** dan rekomendasi
5. **Informasi reproduksi** (seed, versi library)

In [ ]:
# === TEMPLATE LAPORAN PROYEK ===

print('='*70)
print('LAPORAN PROYEK DATA MINING')
print('='*70)

print('''
1. DEFINISI MASALAH
   ─────────────────
   Masalah   : Prediksi churn pelanggan
   Tipe      : Klasifikasi biner (supervised learning)
   Target    : churn (0=stay, 1=churn)
   Metrik    : Balanced Accuracy (karena kelas tidak seimbang)
               + PR-AUC (untuk evaluasi ranking probabilitas)

2. DESKRIPSI DATA
   ───────────────
''')
print(f'   Sampel          : {len(df)}')
print(f'   Fitur numerik   : {len(numeric_features)} -> {numeric_features}')
print(f'   Fitur kategorikal: {len(categorical_features)} -> {categorical_features}')
print(f'   Missing values  : support_score (35), engagement_score (20), channel (10)')
print(f'   Distribusi target: Stay {df["churn"].value_counts()[0]/len(df)*100:.0f}% / '
      f'Churn {df["churn"].value_counts()[1]/len(df)*100:.0f}%')

print('''
3. PREPROCESSING
   ──────────────
   - Numerik: SimpleImputer(median) -> StandardScaler
   - Kategorikal: SimpleImputer(most_frequent) -> OneHotEncoder
   - Semua di dalam Pipeline untuk mencegah data leakage

4. HASIL EKSPERIMEN
   ─────────────────
''')
for name, r in cv_results.items():
    print(f'   {name:25s} -> Bal Acc: {r["bal_acc_mean"]:.3f} ± {r["bal_acc_std"]:.3f}')

print(f'''
   Model terpilih: Random Forest (tuned)
   Parameter     : {grid.best_params_}
   Test Bal Acc   : {balanced_accuracy_score(y_test, y_pred_final):.3f}
   Test PR-AUC    : {average_precision_score(y_test, y_proba_final):.3f}

5. LIMITASI DAN REKOMENDASI
   ────────────────────────
   - Data disimulasikan -> belum divalidasi pada data pelanggan nyata
   - Feature engineering minimal -> fitur domain bisa ditambahkan
   - Threshold belum dioptimalkan untuk cost-benefit spesifik
   - Model perlu dimonitor untuk concept drift setelah deployment

6. REPRODUKSI
   ──────────
   Random State : {RANDOM_STATE}
   sklearn      : {sklearn.__version__}
   numpy        : {np.__version__}
   pandas       : {pd.__version__}
''')

---

## Checklist Proyek Data Mining

Gunakan checklist ini setiap kali mengerjakan proyek:

### Sebelum Modeling
- [ ] Definisi target dan horizon waktu sudah jelas
- [ ] Kolom yang terjadi **setelah** target tidak dipakai (hindari leakage)
- [ ] EDA dilakukan: distribusi, missing, outlier, korelasi
- [ ] Train/test split dilakukan **sebelum** preprocessing apapun

### Saat Modeling
- [ ] Preprocessing ada di dalam Pipeline / ColumnTransformer
- [ ] Baseline model dibuat sebagai pembanding
- [ ] Minimal 2-3 algoritma dibandingkan
- [ ] Cross-validation digunakan (bukan 1x split)
- [ ] class_weight dipertimbangkan jika kelas tidak seimbang

### Setelah Modeling
- [ ] Metrik sesuai konteks bisnis (bukan hanya accuracy)
- [ ] Data test dievaluasi **SATU KALI** di akhir
- [ ] Feature importance dihitung dan diinterpretasi
- [ ] Threshold dipilih berdasarkan kebutuhan (bukan otomatis 0.5)
- [ ] Hasil, parameter, dan seed didokumentasikan
- [ ] Limitasi model diakui secara jujur

---

## Latihan dan Quiz

### Quiz Teori

1. Apa fungsi **ColumnTransformer** dan mengapa kita membutuhkannya?
2. Jelaskan perbedaan antara `SimpleImputer(strategy='mean')` dan `SimpleImputer(strategy='median')`. Kapan masing-masing lebih tepat?
3. Apa yang terjadi jika ada kategori baru di test set yang tidak ada di training set? Bagaimana `handle_unknown='ignore'` menangani ini?
4. Mengapa **silhouette score** bukan satu-satunya cara memilih jumlah cluster?
5. Dalam konteks churn prediction, mengapa **False Negative** (tidak mendeteksi churn) lebih mahal daripada **False Positive** (alarm palsu)?
6. Apa itu **reproducibility** dan mengapa penting dalam data mining?

### Jawaban Quiz

1. ...
2. ...
3. ...
4. ...
5. ...
6. ...

In [ ]:
# LATIHAN 1: Modifikasi pipeline untuk menggunakan GradientBoosting 
# ──────────────────────────────────────────────────────────────────
# Tugas:
#  a) Buat pipeline dengan ColumnTransformer + GradientBoostingClassifier
#  b) Lakukan GridSearchCV dengan param_grid:
#     - n_estimators: [50, 100, 200]
#     - max_depth: [2, 3, 5]
#     - learning_rate: [0.01, 0.1, 0.2]
#  c) Evaluasi pada data test
#  d) Bandingkan hasilnya dengan Random Forest
#
# Tulis kode Anda di bawah:



In [ ]:
# LATIHAN 2: Proyek mini dengan dataset sendiri
# ───────────────────────────────────────────────
# Tugas:
#  a) Pilih dataset CSV dari Kaggle (contoh: Titanic, Heart Disease, dll)
#  b) Buat notebook baru menggunakan template dari level ini:
#     - EDA (Level Light)
#     - Train/Test Split + Pipeline (Level Medium)
#     - GridSearchCV + Feature Importance (Level High)
#     - ColumnTransformer + Laporan (Level Extra High)
#  c) Dokumentasikan setiap keputusan preprocessing
#
# Ini adalah latihan terbuka. Mulai di bawah:



In [ ]:
# LATIHAN 3: Clustering pada data churn (unsupervised)
# ─────────────────────────────────────────────────────
# Tugas:
#  a) Ambil fitur numerik dari df_project (tanpa target churn)
#  b) Standardisasi fitur
#  c) Jalankan K-Means untuk K=2 sampai K=8
#  d) Plot silhouette score
#  e) Bandingkan cluster yang ditemukan dengan label churn asli
#  f) Apakah K-Means bisa menemukan pola churn tanpa label?
#
# Tulis kode Anda di bawah:



---

## Selamat! Anda Telah Menyelesaikan Semua Level!

### Ringkasan Perjalanan Belajar

| Level | Yang Dipelajari |
|:------|:----------------|
|  Light | EDA, kualitas data, visualisasi, fitur vs target |
|  Medium | KNN, Decision Tree, Logistic Regression, pipeline, cross-validation |
|  High | GridSearchCV, class imbalance, ROC/PR curve, Random Forest, feature importance |
|  Extra High | ColumnTransformer, end-to-end pipeline, clustering, threshold tuning, laporan proyek |

### Langkah Selanjutnya

1. **Kerjakan proyek dengan data nyata**  -  ambil CSV dari Kaggle dan terapkan template ini
2. **Pelajari teknik lanjutan**: XGBoost/LightGBM, SMOTE, feature selection
3. **Belajar deployment**: Flask/FastAPI untuk serving model
4. **Jelajahi deep learning**: untuk data non-tabular (gambar, teks)
5. **Baca paper dan buku**: ISLR, Hands-on ML with Scikit-Learn (Géron)

> **Jika Anda sudah mampu menjelaskan tujuan bisnis, sumber data, potensi leakage, metrik, dan limitasi model, Anda sudah melakukan data mining dengan fondasi yang benar.** 